# Step 7: MOS listening test — অডিও তৈরি

এই notebook একই বাক্যের জন্য ৪টা সিস্টেমের অডিও বানায় আর Google Form-এর জন্য একটা ফোল্ডারে রাখে:

1. আসল রেকর্ডিং (V1-এর গলা, মাপকাঠি)
2. MMS-TTS base (`facebook/mms-tts-ben`, noise 0.3)
3. Indic Parler-TTS (Aditi)
4. V1 voice clone (attempt 3, `model_V1_v3/best`, noise 0.3)

**আউটপুট** (`Sylheti_Thesis/voice_clone/mos_test/`):
- `audio_for_listeners/` — `audio_01.wav` … (রেটিং), `sim_01.wav` + `ref_01.wav` … (কণ্ঠ মেলানো)। নাম দেখে সিস্টেম বোঝা যায় না।
- `answer_key_do_not_share/mos_key.csv` — কোন ফাইল কোন সিস্টেম। এটা শ্রোতাদের দেবেন না।

GPU runtime লাগবে। Cell 1 চালানোর পর runtime নিজে restart হবে, তারপর Cell 2 থেকে চালাবেন।
Indic Parler-TTS-এর জন্য Colab Secrets-এ `HF_TOKEN` থাকতে হবে (আগে Step 4b-তে যেটা রেখেছিলেন)।

In [ ]:
# Cell 1: install (একবারই; শেষে runtime নিজে restart হবে)
import os
MARK = '/content/.mos_env_v1'
if not os.path.exists(MARK):
    !pip -q install git+https://github.com/huggingface/parler-tts.git
    !pip -q install "transformers==4.46.1" "protobuf>=6.31.1,<7" soundfile librosa pyloudnorm
    open(MARK, 'w').write('ok')
    print('install শেষ, runtime restart হচ্ছে... এরপর Cell 2 চালান')
    os.kill(os.getpid(), 9)
else:
    print('environment আগেই তৈরি, Cell 2 চালান')

In [ ]:
# Cell 2: Drive mount + Hugging Face login (HF_TOKEN Colab secret থেকে)
import os
os.environ['USE_TF'] = '0'
from google.colab import drive, userdata
drive.mount('/content/drive')
from huggingface_hub import login
login(token=userdata.get('HF_TOKEN'))
print('ready')

In [ ]:
# Cell 3: বাক্য বাছাই + ৪টা সিস্টেমের অডিও তৈরি
import os, re, random, numpy as np, pandas as pd, soundfile as sf, librosa, torch, pyloudnorm as pyln
os.environ['USE_TF'] = '0'
from transformers import VitsModel, AutoTokenizer
from parler_tts import ParlerTTSForConditionalGeneration

# ---------- সেটিং ----------
BASE      = '/content/drive/MyDrive/Sylheti_Thesis/voice_clone'
EVAL_CSV  = f'{BASE}/eval_sentences_V1.csv'           # V1-এর ৫০টা test বাক্য (training-এ ব্যবহার হয়নি)
REAL_DIR  = f'{BASE}/raw_V1/web_recorder_sylheti'     # আসল রেকর্ডিং: syl_<n>.wav
V1_MODEL  = f'{BASE}/model_V1_v3/best'                # attempt 3, step 2,000
OUT       = f'{BASE}/mos_test'
AUDIO_DIR = f'{OUT}/audio_for_listeners'
KEY_DIR   = f'{OUT}/answer_key_do_not_share'
N_SENT    = 10        # কয়টা বাক্য (10 বাক্য x 4 সিস্টেম = 40টা অডিও)
SIM_V1    = N_SENT    # কণ্ঠ মেলানো: V1-এর কয়টা জোড়া
SIM_ANCHOR = 3        # কণ্ঠ মেলানো: MMS আর Parler-এর কয়টা জোড়া (তুলনার জন্য)
SEED      = 2026
SR        = 16000     # সব অডিও একই sample rate-এ, যাতে মান দেখে সিস্টেম চেনা না যায়
NOISE     = 0.3
TARGET_LUFS = -23.0
PARLER_ID = 'ai4bharat/indic-parler-tts'
PARLER_DESC = ("Aditi speaks clearly in standard Bengali at a moderate pace "
               "with a close recording quality and no background noise.")
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

# ---------- আগের ফাইল থাকলে থামবে (কিছু মুছবে না) ----------
if os.path.exists(AUDIO_DIR) and os.listdir(AUDIO_DIR):
    raise SystemExit(f'{AUDIO_DIR} ফোল্ডারে আগে থেকেই ফাইল আছে। '
                     'নতুন করে বানাতে চাইলে Drive-এ পুরনো mos_test ফোল্ডারের নাম বদলে দিন, তারপর আবার চালান।')
os.makedirs(AUDIO_DIR, exist_ok=True); os.makedirs(KEY_DIR, exist_ok=True)
for p in [EVAL_CSV, REAL_DIR, V1_MODEL]:
    assert os.path.exists(p), f'পাওয়া যায়নি: {p}'

# ---------- বাক্য বাছাই ----------
df = pd.read_csv(EVAL_CSV)
df['num'] = df['path'].str.extract(r'syl_(\d+)\.wav')[0].astype(int)
df['real_path'] = df['num'].map(lambda n: f'{REAL_DIR}/syl_{n}.wav')
df['clean'] = df['text'].str.replace(r'["\'“”‘’()\[\]]', '', regex=True).str.strip()
ok = (df['clean'].str.len().between(15, 90)
      & ~df['clean'].str.contains(r'[0-9০-৯A-Za-z]')
      & df['real_path'].map(os.path.exists))
cand = df[ok]
assert len(cand) >= N_SENT, f'যথেষ্ট বাক্য নেই: {len(cand)}'
sel = cand.sample(N_SENT, random_state=SEED).reset_index(drop=True)
sel['sent_id'] = [f'S{i+1:02d}' for i in range(len(sel))]
print(f'{len(sel)}টা বাক্য বাছাই হয়েছে:')
for r in sel.itertuples(): print(' ', r.sent_id, r.excel_row, r.clean)

# ---------- অডিও প্রসেসিং ----------
meter = pyln.Meter(SR)
def finish(wav, sr):
    wav = np.asarray(wav, dtype=np.float32).squeeze()
    if sr != SR: wav = librosa.resample(wav, orig_sr=sr, target_sr=SR)
    wav, _ = librosa.effects.trim(wav, top_db=35)
    wav = np.concatenate([np.zeros(int(0.15*SR), np.float32), wav, np.zeros(int(0.15*SR), np.float32)])
    try:
        wav = pyln.normalize.loudness(wav, meter.integrated_loudness(wav), TARGET_LUFS)
    except Exception:
        pass
    peak = np.abs(wav).max()
    if peak > 0.95: wav = wav / peak * 0.95
    return wav.astype(np.float32)

def vits_tts(model, tok, text, seed):
    torch.manual_seed(seed)
    model.noise_scale = NOISE; model.noise_scale_duration = NOISE
    inp = tok(text, return_tensors='pt').to(device)
    with torch.no_grad():
        w = model(**inp).waveform[0].cpu().numpy()
    return w, model.config.sampling_rate

audio = {}   # (sent_id, system) -> wav

# 1) আসল রেকর্ডিং
for r in sel.itertuples():
    w, sr = librosa.load(r.real_path, sr=None)
    audio[(r.sent_id, 'real')] = finish(w, sr)

# 2) MMS-TTS base
tok = AutoTokenizer.from_pretrained('facebook/mms-tts-ben')
m = VitsModel.from_pretrained('facebook/mms-tts-ben').to(device).eval()
for i, r in enumerate(sel.itertuples()):
    audio[(r.sent_id, 'mms_base')] = finish(*vits_tts(m, tok, r.clean, SEED + i))
del m; torch.cuda.empty_cache()

# 3) V1 voice clone (attempt 3, step 2,000)
tok = AutoTokenizer.from_pretrained(V1_MODEL)
m = VitsModel.from_pretrained(V1_MODEL).to(device).eval()
for i, r in enumerate(sel.itertuples()):
    audio[(r.sent_id, 'v1_clone')] = finish(*vits_tts(m, tok, r.clean, SEED + i))
del m; torch.cuda.empty_cache()

# 4) Indic Parler-TTS
pm = ParlerTTSForConditionalGeneration.from_pretrained(PARLER_ID).to(device).eval()
ptok = AutoTokenizer.from_pretrained(PARLER_ID)
dtok = AutoTokenizer.from_pretrained(pm.config.text_encoder._name_or_path)
desc = dtok(PARLER_DESC, return_tensors='pt').to(device)
for i, r in enumerate(sel.itertuples()):
    torch.manual_seed(SEED + i)
    pr = ptok(r.clean, return_tensors='pt').to(device)
    with torch.no_grad():
        g = pm.generate(input_ids=desc.input_ids, attention_mask=desc.attention_mask,
                        prompt_input_ids=pr.input_ids, prompt_attention_mask=pr.attention_mask)
    audio[(r.sent_id, 'indic_parler')] = finish(g.cpu().numpy(), pm.config.sampling_rate)
del pm; torch.cuda.empty_cache()
print('৪টা সিস্টেমের অডিও তৈরি:', len(audio))

# ---------- রেটিং অংশ: এলোমেলো নাম audio_01.wav ... ----------
rng = random.Random(SEED)
SYSTEMS = ['real', 'mms_base', 'indic_parler', 'v1_clone']
rating = [(s, sysname) for s in sel['sent_id'] for sysname in SYSTEMS]
rng.shuffle(rating)
key_rows = []
text_of = dict(zip(sel['sent_id'], sel['clean']))
for k, (sid, sysname) in enumerate(rating, 1):
    fn = f'audio_{k:02d}.wav'
    sf.write(f'{AUDIO_DIR}/{fn}', audio[(sid, sysname)], SR)
    key_rows.append(dict(file=fn, task='rating', system=sysname, sent_id=sid, ref_file='', text=text_of[sid]))

# ---------- কণ্ঠ মেলানো অংশ: sim_XX.wav + ref_XX.wav ----------
ids = list(sel['sent_id'])
pairs = [(s, 'v1_clone') for s in ids[:SIM_V1]]
pairs += [(s, 'mms_base') for s in rng.sample(ids, SIM_ANCHOR)]
pairs += [(s, 'indic_parler') for s in rng.sample(ids, SIM_ANCHOR)]
rng.shuffle(pairs)
# রেফারেন্স = একই বক্তার আসল রেকর্ডিং, কিন্তু অন্য একটা বাক্যের।
# এতে শ্রোতা কথা মিলিয়ে নয়, শুধু গলা মিলিয়ে নম্বর দেবে।
other = {sid: ids[(i + 1) % len(ids)] for i, sid in enumerate(ids)}
for k, (sid, sysname) in enumerate(pairs, 1):
    fn, rf = f'sim_{k:02d}.wav', f'ref_{k:02d}.wav'
    sf.write(f'{AUDIO_DIR}/{fn}', audio[(sid, sysname)], SR)
    sf.write(f'{AUDIO_DIR}/{rf}', audio[(other[sid], 'real')], SR)
    key_rows.append(dict(file=fn, task='similarity', system=sysname, sent_id=sid,
                         ref_file=rf, ref_sent_id=other[sid], text=text_of[sid]))

key = pd.DataFrame(key_rows)
key.to_csv(f'{KEY_DIR}/mos_key.csv', index=False, encoding='utf-8-sig')
sel[['sent_id', 'excel_row', 'clean', 'real_path']].to_csv(f'{KEY_DIR}/mos_sentences.csv', index=False, encoding='utf-8-sig')

print(f"\nরেটিং অডিও: {(key.task=='rating').sum()}টা, কণ্ঠ মেলানোর জোড়া: {(key.task=='similarity').sum()}টা")
print(key.groupby(['task', 'system']).size())
print('\nফোল্ডার:', AUDIO_DIR)
print('উত্তরপত্র (শ্রোতাদের দেবেন না):', f'{KEY_DIR}/mos_key.csv')
print('\nএখন Drive-এ audio_for_listeners ফোল্ডারটা খুলুন, ব্রাউজারের ঠিকানা থেকে folder ID কপি করে Apps Script-এ বসান।')

In [ ]:
# Cell 4 (ঐচ্ছিক): কয়েকটা অডিও শুনে দেখুন
from IPython.display import Audio, display
import pandas as pd
key = pd.read_csv(f'{KEY_DIR}/mos_key.csv')
for sysname in ['real', 'mms_base', 'indic_parler', 'v1_clone']:
    row = key[(key.task == 'rating') & (key.system == sysname)].iloc[0]
    print(sysname, '|', row.sent_id, row.text)
    display(Audio(f"{AUDIO_DIR}/{row.file}"))